# Wikipedia attention: Week 2

Week 2 tasks for the Wikipedia side (James). All logic lives in `src/`; this notebook only calls it and
shows the results. Run `python -m src.wiki_pipeline` first (pulls and caches the data), then
**Restart and Run All**.

**Holdout rule:** research (spike returns, edit timing, strategy) uses data before 2025-10-01 only.
The last-3-weeks check is the one place recent data is used, and only for looking.

In [1]:
import sys

sys.path.append("..")

import pandas as pd

from src import config
from src import wiki_analysis as wa
from src.trends_compare import cached_series, gdelt_daily, tracking_correlation, trendspy_weekly
from src.wiki_attention import weekly
from src.wiki_edits import daily_edits
from src.wiki_pipeline import PROCESSED, RAW, ROOT, load_edits, shared_path, universe, windows
from src.wiki_prices import research_only

pd.set_option("display.width", 160, "display.max_columns", 30)
MARKET = "US"
TRACKED = config.tickers(MARKET)          # the 10 stocks shared with Joshua
ALL = list(universe(MARKET))              # + the strategy universe (51 stocks)
HOLDOUT = pd.Timestamp(config.holdout_start())
BOOT = config.load()["wikipedia"]["bootstrap"]
WIN = windows()
OUT = ROOT / "results" / "wikipedia"
OUT.mkdir(parents=True, exist_ok=True)

## Step 1. Data coverage after the re-pull

Research views run Jul 1, 2015 to Sep 30, 2025; the recent year sits in separate `views_recent/` files.
Each article's views include all redirects to it, so renamed articles (Facebook, Inc. -> Meta Platforms,
Tesla Motors -> Tesla, Inc.) keep their history.

In [2]:
rows = []
for t in ALL:
    r = pd.read_csv(RAW / "views_research" / f"{t}.csv", index_col=0, parse_dates=True).iloc[:, 0]
    c = pd.read_csv(RAW / "views_recent" / f"{t}.csv", index_col=0, parse_dates=True).iloc[:, 0]
    revs = pd.read_csv(RAW / "revisions_research" / "en" / f"{t}.csv")
    rows.append({"ticker": t, "research_first": r.index.min().date(), "research_last": r.index.max().date(),
                 "recent_first": c.index.min().date(), "recent_last": c.index.max().date(),
                 "research_views_m": r.sum() / 1e6, "zero_view_days": int((r == 0).sum()),
                 "research_edits": int((~revs["before_start"]).sum())})
coverage = pd.DataFrame(rows).set_index("ticker")
assert (coverage["research_last"] < HOLDOUT.date()).all() and (coverage["recent_first"] >= HOLDOUT.date()).all()
assert (coverage["research_first"] == pd.Timestamp("2015-07-01").date()).all()
coverage.to_csv(OUT / "coverage.csv")
coverage.loc[TRACKED].round(1)

,research_first,research_last,recent_first,recent_last,research_views_m,zero_view_days,research_edits
ticker,,,,,,,
AAPL,2015-07-01,2025-09-30,2025-10-01,2026-10-25,47.6,0,3372
MSFT,2015-07-01,2025-09-30,2025-10-01,2026-10-25,32.3,0,1962
NVDA,2015-07-01,2025-09-30,2025-10-01,2026-10-25,13.6,0,2207
TSLA,2015-07-01,2025-09-30,2025-10-01,2026-10-25,33.5,0,8016
AMZN,2015-07-01,2025-09-30,2025-10-01,2026-10-25,60.6,0,3431
META,2015-07-01,2025-09-30,2025-10-01,2026-10-25,28.7,0,1644
GOOGL,2015-07-01,2025-09-30,2025-10-01,2026-10-25,21.0,0,2901
AMD,2015-07-01,2025-09-30,2025-10-01,2026-10-25,8.4,0,1539
NFLX,2015-07-01,2025-09-30,2025-10-01,2026-10-25,56.9,0,7519


In [3]:
# Wikipedia views by year for the tracked stocks (millions). META's company article only starts in late 2019.
yearly = pd.concat({t: pd.read_csv(RAW / "views_research" / f"{t}.csv", index_col=0, parse_dates=True).iloc[:, 0]
                    for t in TRACKED}, axis=1)
(yearly.groupby(yearly.index.year).sum() / 1e6).round(2)

,AAPL,MSFT,NVDA,TSLA,AMZN,META,GOOGL,AMD,NFLX,JPM
date,,,,,,,,,,
2015,2.65,1.47,0.30,1.43,2.07,0.02,1.47,0.33,1.94,0.51
2016,4.98,2.97,0.67,3.17,4.13,0.20,2.22,0.65,3.67,1.14
2017,4.51,2.71,0.79,3.31,5.56,0.06,2.15,0.80,3.41,1.15
2018,4.81,2.65,0.78,3.25,5.42,0.06,2.07,0.82,4.29,1.21
2019,4.73,3.01,0.65,2.89,5.45,0.23,2.27,0.82,5.24,1.25
2020,5.03,3.27,1.07,3.64,6.88,2.55,2.76,1.13,6.74,1.16
2021,4.47,3.76,0.96,4.23,9.73,6.73,2.03,0.80,8.15,1.16
2022,4.66,4.08,1.04,3.68,8.88,8.44,1.67,0.74,9.38,1.13
2023,5.07,3.29,1.68,3.02,5.42,6.07,1.56,0.72,6.22,1.38


## Step 2. Spike flags, past weeks only

`flag_spikes` now takes the robust z-score's median and MAD from earlier weeks only (26 needed), the same
rule as Joshua's Reddit side (`config.yaml` `spike`). The look-ahead test
(`tests/test_wiki_attention.py::test_changing_future_weeks_leaves_past_flags_unchanged`) changes only weeks
after t and checks every flag up to t is unchanged.

In [4]:
att_r = pd.read_csv(shared_path("research"))
att_c = pd.read_csv(shared_path("recent"))
assert pd.to_datetime(att_r["week"]).max() + pd.Timedelta(days=6) < HOLDOUT
spike_counts = (att_r[att_r["ticker"].isin(TRACKED)].groupby(["ticker", "source"])["spike"].agg(["sum", "count"])
                .unstack("source"))
spike_counts

sum                           count                
source wikipedia_edits wikipedia_views wikipedia_edits wikipedia_views
ticker                                                                
AAPL                13              21             534             534
AMD                  5              10             534             534
AMZN                13               9             534             534
GOOGL               14              22             534             534
JPM                 11               7             534             534
META                19              57             534             534
MSFT                10              13             534             534
NFLX                 9              15             534             534
NVDA                 9              37             534             534
TSLA                22              32             534             534

## Step 3. Returns 1-4 weeks after a spike

Spike weeks vs all other weeks, research period only. `ret_next{k}` runs from the last close of the spike
week to k weeks later; abnormal returns subtract beta x SPY (52-week beta from earlier weeks).
95% CIs resample whole weeks (1,000 times), because stocks move together within a week.

In [5]:
prices = pd.read_csv(PROCESSED / "weekly_prices.csv", parse_dates=["week"])
prices_r = research_only(prices, HOLDOUT)
att_rd = att_r.assign(week=pd.to_datetime(att_r["week"]))
spike_ret = pd.concat([
    wa.spike_returns(att_rd, prices_r, "wikipedia_views", TRACKED, **BOOT).assign(universe="10 tracked"),
    wa.spike_returns(att_rd, prices_r, "wikipedia_views", None, **BOOT).assign(universe="51 stocks"),
    wa.spike_returns(att_rd, prices_r, "wikipedia_edits", None, **BOOT).assign(universe="51 stocks"),
], ignore_index=True)
spike_ret.to_csv(OUT / "spike_returns.csv", index=False)
show = spike_ret.copy()
for c in ["mean_spike", "mean_normal", "diff", "ci_low", "ci_high"]:
    show[c] = (100 * show[c]).round(2)   # percent
show[["universe", "source", "value", "n_spike", "n_normal", "mean_spike", "mean_normal", "diff", "ci_low", "ci_high"]]

,universe,source,value,n_spike,n_normal,mean_spike,mean_normal,diff,ci_low,ci_high
0,10 tracked,wikipedia_views,ret_next1,223,5107,1.18,0.68,0.50,-0.29,1.26
1,10 tracked,wikipedia_views,ret_next2,223,5097,1.92,1.37,0.55,-0.56,1.56
2,10 tracked,wikipedia_views,ret_next3,222,5088,2.32,2.08,0.25,-1.16,1.75
3,10 tracked,wikipedia_views,ret_next4,222,5078,2.94,2.79,0.15,-1.85,2.17
4,10 tracked,wikipedia_views,abn_ret_next1,223,5107,0.50,0.33,0.16,-0.46,0.81
5,10 tracked,wikipedia_views,abn_ret_next2,223,5097,0.84,0.67,0.17,-0.84,1.15
6,10 tracked,wikipedia_views,abn_ret_next3,222,5088,0.87,1.01,-0.14,-1.33,1.10
7,10 tracked,wikipedia_views,abn_ret_next4,222,5078,0.80,1.36,-0.56,-2.19,1.05
8,51 stocks,wikipedia_views,ret_next1,734,26449,0.65,0.38,0.27,-0.09,0.63
9,51 stocks,wikipedia_views,ret_next2,733,26399,1.18,0.76,0.42,-0.08,0.95


## Step 5. Edit-summary analysis

Per company per week: edits, distinct editors, bytes added/removed, reverts and keyword tags
(keywords in `config.yaml` `wikipedia.edit_tags`).

In [6]:
we_r = pd.read_csv(PROCESSED / "weekly_edits_research.csv", parse_dates=["week"])
tag_cols = [c for c in we_r.columns if c.startswith("tag_")]
per_week = we_r[we_r["ticker"].isin(TRACKED)].groupby("ticker")[["edits", "editors", "reverts", *tag_cols]].mean()
per_week.round(2)

,edits,editors,reverts,tag_earnings,tag_ceo_executive,tag_legal,tag_acquisition_merger,tag_product_launch,tag_layoff,tag_stock_shares
ticker,,,,,,,,,,
AAPL,6.31,3.81,0.41,0.14,0.08,0.04,0.06,0.17,0.01,0.06
AMD,2.88,1.97,0.42,0.01,0.01,0.01,0.10,0.13,0.00,0.02
AMZN,6.40,4.34,0.66,0.04,0.07,0.05,0.05,0.15,0.00,0.02
GOOGL,5.43,3.73,1.26,0.06,0.10,0.08,0.03,0.06,0.00,0.04
JPM,3.76,2.10,0.40,0.01,0.04,0.05,0.05,0.01,0.00,0.01
META,3.08,2.12,0.42,0.11,0.06,0.09,0.05,0.02,0.06,0.06
MSFT,3.66,2.44,0.31,0.03,0.05,0.01,0.20,0.04,0.03,0.02
NFLX,14.05,9.72,2.62,0.04,0.03,0.08,0.11,0.23,0.01,0.04
NVDA,4.13,2.62,0.52,0.02,0.02,0.02,0.26,0.14,0.00,0.04


In [7]:
# Which tags show up more in view-spike weeks? (lift = tagged edits per week in spike weeks / other weeks)
lift = wa.tag_lift(we_r, att_rd)
lift.to_csv(OUT / "tag_lift.csv")
lift.round(2)

,normal_weeks,spike_weeks,lift,total
tag_layoff,0.00,0.02,5.03,106
tag_ceo_executive,0.03,0.13,3.96,979
tag_acquisition_merger,0.07,0.23,3.53,1903
tag_stock_shares,0.02,0.06,2.75,635
tag_legal,0.03,0.08,2.63,910
tag_earnings,0.02,0.05,2.46,563
tag_product_launch,0.06,0.13,2.33,1577
edits,3.13,7.25,2.31,88352
reverts,0.45,0.88,1.98,12466


In [8]:
# Tone of edit summaries (VADER, as in Joshua's scorers)
edits_tracked = {t: load_edits(t, universe(MARKET)[t], WIN) for t in TRACKED}
research_edits = pd.concat([e[e["timestamp"] < HOLDOUT] for e in edits_tracked.values()])
tone = wa.summary_tone(research_edits["comment"])
{k: round(v, 3) for k, v in tone.items()}

{'n_summaries': 27669,
 'share_empty': 0.2,
 'share_neutral': 0.729,
 'share_positive': 0.166,
 'share_negative': 0.105}

In [9]:
# Timing: edit spikes around view spikes and big price moves (lag < 0: edits came first)
timing = pd.concat([wa.edit_timing(att_rd[att_rd["ticker"].isin(TRACKED)], prices_r).assign(universe="10 tracked"),
                    wa.edit_timing(att_rd, prices_r).assign(universe="51 stocks")])
timing.to_csv(OUT / "edit_timing.csv", index=False)
timing.pivot_table(index=["universe", "event", "signal"], columns="lag_weeks", values="lift").round(2)

lag_weeks                                -2    -1     0     1     2
universe   event          signal                                   
10 tracked big price move edits spike  0.64  0.79  2.06  0.63  1.42
                          views spike  1.17  0.80  1.95  0.98  0.98
           views spike    edits spike  0.96  1.92  2.87  0.96  1.34
51 stocks  big price move edits spike  1.11  1.06  1.72  1.11  1.03
                          views spike  1.22  1.43  3.10  1.67  1.38
           views spike    edits spike  1.35  2.41  3.76  1.06  0.96

In [10]:
# Within a view-spike week: day of most edits minus day of most views
lags = []
for t in TRACKED:
    v = pd.read_csv(RAW / "views_research" / f"{t}.csv", index_col=0, parse_dates=True).iloc[:, 0]
    e = edits_tracked[t]
    d = daily_edits(e[e["timestamp"] < HOLDOUT], v.index)
    wk = pd.to_datetime(att_r[(att_r["ticker"] == t) & (att_r["source"] == "wikipedia_views") & att_r["spike"]]["week"])
    lags.append(wa.daily_peak_lag(v, d, list(wk)))
lags = pd.concat(lags)
print(len(lags), "view-spike weeks with edits nearby; median lag", lags.median(), "days")
lags.value_counts().sort_index()

198 view-spike weeks with edits nearby; median lag 0.0 days


edit_peak_minus_view_peak_days
-7.000000    10
-6.000000     9
-5.000000     6
-4.500000     2
-4.000000     6
-3.333333     1
-3.000000    11
-2.666667     1
-2.250000     1
-2.000000    11
-1.800000     1
-1.666667     1
-1.500000     5
-1.333333     1
-1.000000    17
-0.500000     3
-0.400000     1
 0.000000    19
 0.444444     1
 0.500000     1
 0.666667     1
 1.000000    13
 1.250000     1
 1.500000     3
 1.666667     1
 2.000000    12
 2.166667     1
 2.333333     1
 2.666667     1
 3.000000    10
 3.500000     1
 4.000000    16
 4.500000     2
 5.000000     7
 6.000000     8
 6.500000     2
 7.000000    10
Name: count, dtype: int64

## Step 6. Last-3-weeks spike check (looking only)

Weeks starting Sep 28, Oct 5 and Oct 12, 2026. Wikipedia views and edits columns for the table Joshua
builds (his `spike_check_recent.py` joins on ticker and week), plus the price move that week and the next.

In [11]:
LAST_WEEKS = ["2026-09-28", "2026-10-05", "2026-10-12"]
last3 = wa.last_weeks_table(att_c, prices.assign(week=prices["week"].dt.strftime("%Y-%m-%d")), LAST_WEEKS, TRACKED)
last3.to_csv(OUT / "last3weeks_wikipedia.csv", index=False)
show = last3.copy()
for c in ["ret_week", "ret_next_week", "abn_ret_week"]:
    show[c] = (100 * show[c]).round(1)
show.round(2)

,ticker,market,week,wiki_views,wiki_views_ratio,wiki_views_rz,wiki_views_spike,wiki_edits,wiki_edits_ratio,wiki_edits_rz,wiki_edits_spike,ret_week,ret_next_week,abn_ret_week
0,AAPL,US,2026-09-28,45188.0,0.89,-0.88,False,1.0,0.50,-0.92,False,-2.2,0.9,-2.0
1,AAPL,US,2026-10-05,43447.0,0.85,-1.21,False,3.0,1.14,0.18,False,0.9,2.0,-0.1
2,AAPL,US,2026-10-12,41474.0,0.81,-1.59,False,1.0,0.50,-0.92,False,2.0,1.2,3.6
3,AMD,US,2026-09-28,14028.0,1.53,2.96,True,2.0,1.00,0.00,False,0.5,-4.1,0.9
4,AMD,US,2026-10-05,12078.0,1.31,1.93,False,5.0,1.71,0.52,False,-4.1,-13.0,-6.2
5,AMD,US,2026-10-12,11276.0,1.21,1.33,False,0.0,0.25,-1.35,False,-13.0,-2.3,-8.3
6,AMZN,US,2026-09-28,55539.0,0.98,-0.09,False,2.0,0.86,-0.20,False,0.7,4.3,1.1
7,AMZN,US,2026-10-05,57996.0,1.02,0.30,False,1.0,0.67,-0.54,False,4.3,-4.7,2.3
8,AMZN,US,2026-10-12,57839.0,1.02,0.27,False,3.0,1.60,0.62,False,-4.7,-0.8,-1.1
9,GOOGL,US,2026-09-28,22023.0,1.05,0.43,False,0.0,1.00,0.00,False,-0.1,2.4,0.2


In [12]:
# What the editors wrote in spiking stock-weeks (to read before writing the 2-3 sentences per stock)
spiking = last3[last3["wiki_views_spike"] | last3["wiki_edits_spike"]]
for r in spiking.itertuples():
    w = pd.Timestamp(r.week)
    e = edits_tracked[r.ticker]
    e = e[(e["timestamp"] >= w) & (e["timestamp"] < w + pd.Timedelta(days=7)) & ~e["revert"]]
    print(f"\n{r.ticker} week of {r.week}: views {r.wiki_views:.0f} ({r.wiki_views_ratio:.1f}x), "
          f"edits {r.wiki_edits:.0f} ({r.wiki_edits_ratio:.1f}x)")
    for c in e["comment"].str.slice(0, 110).loc[lambda s: s.str.len() > 0].drop_duplicates().head(6):
        print("   -", c)


AMD week of 2026-09-28: views 14028 (1.5x), edits 2 (1.0x)
   - /* Acquisition history */ World Labs

TSLA week of 2026-10-12: views 44799 (2.1x), edits 4 (0.8x)
   - Updated 2nd-gen Roadster with Oct 2026 re-reveal (TechCrunch, Jalopnik); copyedits (grammar, capitalization, m
   - /* Tesla next-generation vehicle */ removed double link in subsection
   - ce
   - Marked a citation as subscription-restricted


## Step 7. Google Trends alternatives

How well each free source tracks AAPL's weekly Wikipedia views (research weeks). trendspy is used for
exploring only; GDELT is free but rate-limited (one request per 5 seconds), so results are cached.

In [13]:
TRENDS = RAW.parent / "trends"
aapl_views = pd.read_csv(RAW / "views_research" / "AAPL.csv", index_col=0, parse_dates=True).iloc[:, 0]
track = {}
gt = cached_series(TRENDS / "trendspy_AAPL_Apple.csv",
                   lambda: trendspy_weekly("Apple", pd.Timestamp("2020-10-01"), pd.Timestamp("2025-09-28")))
track["Google Trends via trendspy ('Apple')"] = tracking_correlation(gt, aapl_views)
try:
    gd = cached_series(TRENDS / "gdelt_AAPL.csv",
                       lambda: gdelt_daily('("Apple Inc" OR "Tim Cook" OR iPhone)',
                                           pd.Timestamp("2017-01-02"), pd.Timestamp("2025-09-28")))
    track["GDELT news volume"] = tracking_correlation(weekly(gd), aapl_views)
except RuntimeError as err:   # GDELT refuses bursts of requests
    print("GDELT not available right now:", str(err)[:80])
pd.DataFrame(track).T.round(2)

,weeks,corr_level,corr_change,corr_level_spearman
Google Trends via trendspy ('Apple'),261.0,0.2,0.59,0.12
GDELT news volume,454.0,0.2,0.54,0.25


In [14]:
alternatives = pd.DataFrame([
    ["Wikipedia page views", "yes (CC0 data, documented API)", "yes, Jul 2015", "yes: hi/zh Wikipedia, but smaller audiences", "free", "the reference"],
    ["GDELT news volume and tone", "yes (open data)", "yes, Jan 2017 (DOC API)", "yes, many countries and languages", "free, rate-limited", "see table above"],
    ["trendspy", "no: scrapes Google's site", "yes, 2004 (weekly only in <=5-year windows)", "yes", "free", "see table above"],
    ["SerpApi Google Trends", "yes (paid API)", "yes, 2004", "yes", "from about $25/month for 1,000 searches", "same data as trendspy"],
    ["Glimpse", "yes (paid)", "yes", "yes", "no public API price (tool from ~$99/month)", "modelled 'absolute' volumes"],
], columns=["option", "legal to use", "history before 2023", "India and Hong Kong", "cost", "tracks AAPL views"])
alternatives

,option,legal to use,history before 2023,India and Hong Kong,cost,tracks AAPL views
0,Wikipedia page views,"yes (CC0 data, documented API)","yes, Jul 2015","yes: hi/zh Wikipedia, but smaller audiences",free,the reference
1,GDELT news volume and tone,yes (open data),"yes, Jan 2017 (DOC API)","yes, many countries and languages","free, rate-limited",see table above
2,trendspy,no: scrapes Google's site,"yes, 2004 (weekly only in <=5-year windows)",yes,free,see table above
3,SerpApi Google Trends,yes (paid API),"yes, 2004",yes,"from about $25/month for 1,000 searches",same data as trendspy
4,Glimpse,yes (paid),yes,yes,no public API price (tool from ~$99/month),modelled 'absolute' volumes


## Step 11. Strategy: Wiki Attention Breakout (Wikipedia-only)

Rules fixed before the first run (`src/wiki_strategy.py`): views spike + edit burst (>= 2x the 8-week
median) + price above its 50-day average; buy at the next day's close; exit after 4 weeks, an 8% fall,
or a close below the 50-day average; equal weight, max 10% per stock, 50% exposure when SPY is below
its 200-day average; 10 bps costs. 51 stocks, train 2016-2019, test 2020 to Sep 2025, 200 random portfolios.

In [15]:
from src.wiki_strategy import run

res = run(MARKET)
table = pd.DataFrame({k: res[k] for k in ("train", "test")}).T
print(res["decision"], "-", res["why"])
table[["n_trades", "cagr", "sharpe", "max_drawdown", "turnover", "index_cagr", "index_sharpe", "ew_cagr",
       "ew_sharpe", "random_median_sharpe", "pct_random_beaten"]].astype(float).round(3)

Keep on watch - beat >= 90% of random portfolios in test only


,n_trades,cagr,sharpe,max_drawdown,turnover,index_cagr,index_sharpe,ew_cagr,ew_sharpe,random_median_sharpe,pct_random_beaten
train,54.0,-0.007,-0.319,-0.054,2.891,0.143,1.105,0.237,1.556,0.368,2.0
test,92.0,0.033,0.938,-0.057,3.306,0.152,0.779,0.196,0.957,0.408,96.0


In [16]:
res["trades"]["reason"].value_counts()

reason
time             83
stop_or_trend    63
Name: count, dtype: int64

## Findings

See `docs/wikipedia/week2_report.md` for the written summary of these results.